# Score-Based Camera Placement in a Mall

**End-to-end pipeline:**
1. Toy mall geometry with SDF channels
2. Synthetic pedestrian trajectory generation (5 behavioural patterns)
3. KDE intensity field construction per time window
4. Geometry-conditioned diffusion model (ScoreUNet) training
5. Tweedie mean + DDIM posterior sampling at inference
6. Greedy camera initialisation — joint position + orientation
7. Gradient-based refinement via differentiable MC void probability
8. Simulation study + evaluation figures

> **Key idea:** Replace INLA (used in the paper) with a learned score-based model that acts as a data-driven prior over pedestrian intensity fields. The posterior mean feeds the Jensen lower bound; posterior samples close the Jensen gap and give the true void probability.

In [ ]:
"""
Score-Based Camera Placement in a Mall
=======================================
Full end-to-end pipeline:
  1. Toy mall geometry + SDF channels
  2. Synthetic pedestrian trajectory generation
  3. KDE intensity field construction per time window
  4. Geometry-conditioned diffusion model (ScoreUNet) training
  5. Tweedie mean + DDIM posterior sampling at inference
  6. Greedy camera initialisation (position + orientation)
  7. Gradient-based refinement via differentiable void probability
  8. Simulation study + evaluation figures
"""

import math, random, os
from math import pi, exp, inf
from dataclasses import dataclass
from typing import List, Tuple
from collections import Counter

import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import torch
import torch.nn as nn
import torch.nn.functional as F

# ── reproducibility ────────────────────────────────────────────
torch.manual_seed(42); np.random.seed(42); random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## Section 0 — Global Constants

In [ ]:
MALL_SIZE   = 20.0          # 20 m × 20 m
HALF        = MALL_SIZE / 2
RESOLUTION  = 1.0           # 1 m per grid cell
H = W       = int(MALL_SIZE / RESOLUTION)   # 20 × 20 grid
DX2         = RESOLUTION ** 2

# Diffusion schedule
T_DIFF      = 300
BETA_START  = 1e-4
BETA_END    = 0.02
betas       = torch.linspace(BETA_START, BETA_END, T_DIFF)
ALPHA_BAR   = torch.cumprod(1.0 - betas, dim=0)   # ᾱ_t

# Training
N_WINDOWS   = 120
N_EPOCHS    = 15
BATCH_SIZE  = 8
LR          = 2e-4

# Camera model
MAX_RANGE   = 8.0           # metres
FOV_DEG     = 90.0          # half-angle field of view
RHO         = 0.95          # peak detection probability
N_CAMERAS   = 3

# Output directory
OUT_DIR = '/mnt/user-data/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

## Section 1 — Mall Geometry

In [ ]:
def build_geometry() -> Tuple[torch.Tensor, np.ndarray, np.ndarray]:
    """
    Returns geometry tensor [4, H, W] and grid-centre arrays.
      ch0 : SDF    — signed distance to nearest wall, normalised to [0,1]
      ch1 : grad_x — x-gradient of SDF
      ch2 : grad_y — y-gradient of SDF
      ch3 : zones  — 5-zone room map normalised to [0,1]
    """
    xs = np.linspace(-HALF + RESOLUTION/2, HALF - RESOLUTION/2, W)
    ys = np.linspace(-HALF + RESOLUTION/2, HALF - RESOLUTION/2, H)
    XX, YY = np.meshgrid(xs, ys)

    sdf = (np.minimum(HALF - np.abs(XX),
                      HALF - np.abs(YY)) / HALF).astype(np.float32)
    gy, gx = np.gradient(sdf)

    zone = np.zeros((H, W), dtype=np.float32)
    zone[YY >  3] = 1                               # north zone
    zone[YY < -3] = 2                               # south zone
    zone[(XX >  3) & (YY >= -3) & (YY <= 3)] = 3   # east zone
    zone[(XX < -3) & (YY >= -3) & (YY <= 3)] = 4   # west zone

    geo = np.stack([sdf,
                    gx.astype(np.float32),
                    gy.astype(np.float32),
                    zone / 4.0], axis=0)
    return torch.tensor(geo, dtype=torch.float32), xs, ys


GEOMETRY, XS, YS = build_geometry()
SDF_MAP          = GEOMETRY[0]                            # [H, W]
XX_g, YY_g       = np.meshgrid(XS, YS)
GRID_CENTRES     = torch.tensor(
    np.stack([XX_g.ravel(), YY_g.ravel()], axis=1),
    dtype=torch.float32
)  # [H*W, 2]

DOORS = {
    'N': np.array([ 0.0,  HALF - 0.5]),
    'S': np.array([ 0.0, -HALF + 0.5]),
    'E': np.array([ HALF - 0.5,  0.0]),
    'W': np.array([-HALF + 0.5,  0.0]),
}


def world_to_idx(x: float, y: float) -> Tuple[int, int]:
    i = int(np.clip((y + HALF) / RESOLUTION, 0, H - 1))
    j = int(np.clip((x + HALF) / RESOLUTION, 0, W - 1))
    return i, j

## Section 2 — Synthetic Trajectory Generation

In [ ]:
@dataclass
class Trajectory:
    points:  np.ndarray   # [T, 2]  sequence of (x, y) waypoints
    pattern: str


PATTERN_WEIGHTS = {'NS': .30, 'EW': .25, 'diagonal': .20,
                   'central': .15, 'corner': .10}

def _noise(sigma: float = 0.6) -> np.ndarray:
    return np.random.normal(0, sigma, 2)

def _interp(start: np.ndarray, end: np.ndarray,
            n: int = 20) -> np.ndarray:
    ts    = np.linspace(0, 1, n)[:, None]
    path  = start + ts * (end - start)
    path += np.random.normal(0, 0.15, path.shape)
    return path


def generate_trajectory() -> Trajectory:
    pat = np.random.choice(list(PATTERN_WEIGHTS.keys()),
                           p=list(PATTERN_WEIGHTS.values()))
    if pat == 'NS':
        sx  = np.random.normal(0, 1.0)
        pts = _interp(DOORS['N'] + _noise() + [sx, 0],
                      DOORS['S'] + _noise() + [sx, 0])
    elif pat == 'EW':
        sy  = np.random.normal(0, 1.0)
        pts = _interp(DOORS['E'] + _noise() + [0, sy],
                      DOORS['W'] + _noise() + [0, sy])
    elif pat == 'diagonal':
        pts = _interp(DOORS['N'] + _noise(),
                      np.array([HALF - 1., -HALF + 1.]) + _noise())
    elif pat == 'central':
        d   = DOORS[np.random.choice(['N', 'S', 'E', 'W'])]
        mid = _noise(0.5)
        pts = np.vstack([_interp(d + _noise(), mid, 10),
                         _interp(mid, d + _noise(), 10)])
    else:   # corner browser
        mid = np.array([ HALF - 1.,  HALF - 1.]) + _noise()
        end = np.array([ HALF - 1., -HALF + 1.]) + _noise()
        pts = np.vstack([_interp(DOORS['E'] + _noise(), mid, 10),
                         _interp(mid, end, 10)])
    pts = np.clip(pts, -HALF + 0.1, HALF - 0.1)
    return Trajectory(points=pts, pattern=pat)


def generate_window(n_mean: int = 40) -> List[Trajectory]:
    n = max(np.random.poisson(n_mean), 5)
    return [generate_trajectory() for _ in range(n)]

## Section 3 — KDE Intensity Field

In [ ]:
def build_intensity_field(window: List[Trajectory],
                           bandwidth: float = 1.5) -> torch.Tensor:
    """
    Deposit each trajectory point as a Gaussian kernel on the [H,W] grid.
    Mask by SDF (no intensity inside walls), then L1-normalise.
    Returns [1, H, W].
    """
    field = np.zeros((H, W), dtype=np.float32)
    kr    = int(np.ceil(2 * bandwidth / RESOLUTION))

    for traj in window:
        for (x, y) in traj.points:
            ci = int((y + HALF) / RESOLUTION)
            cj = int((x + HALF) / RESOLUTION)
            for di in range(-kr, kr + 1):
                for dj in range(-kr, kr + 1):
                    ni, nj = ci + di, cj + dj
                    if 0 <= ni < H and 0 <= nj < W:
                        d2 = (di * RESOLUTION) ** 2 + (dj * RESOLUTION) ** 2
                        field[ni, nj] += exp(-d2 / (2 * bandwidth ** 2))

    field *= (SDF_MAP.numpy() > 0).astype(np.float32)
    total  = field.sum()
    if total > 0:
        field /= total
    return torch.tensor(field, dtype=torch.float32).unsqueeze(0)   # [1,H,W]


print('Building intensity fields...')
ALL_WINDOWS   = [generate_window() for _ in range(N_WINDOWS)]
ALL_FIELDS    = [build_intensity_field(w) for w in ALL_WINDOWS]
FIELDS_TENSOR = torch.stack(ALL_FIELDS, dim=0)   # [N, 1, H, W]
print(f'  Fields tensor: {FIELDS_TENSOR.shape}')

## Section 4 — ScoreUNet Architecture

In [ ]:
def _gn(ch: int) -> nn.GroupNorm:
    return nn.GroupNorm(min(8, ch), ch)


class TimestepEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim  = dim
        self.proj = nn.Sequential(
            nn.Linear(dim, dim * 4), nn.SiLU(),
            nn.Linear(dim * 4, dim * 4)
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half  = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) *
            torch.arange(half, device=t.device) / half
        )
        args = t[:, None].float() * freqs[None]
        emb  = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return self.proj(emb)   # [B, dim*4]


class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.conv1     = nn.Conv2d(in_ch,  out_ch, 3, padding=1)
        self.conv2     = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm1     = _gn(in_ch)
        self.norm2     = _gn(out_ch)
        self.time_proj = nn.Linear(time_dim, out_ch * 2)
        self.skip      = (nn.Conv2d(in_ch, out_ch, 1)
                          if in_ch != out_ch else nn.Identity())
        self.act       = nn.SiLU()

    def forward(self, x: torch.Tensor,
                te: torch.Tensor) -> torch.Tensor:
        h           = self.act(self.norm1(x))
        h           = self.conv1(h)
        scale, shift = self.time_proj(te).chunk(2, dim=-1)
        h           = (self.norm2(h)
                       * (1 + scale[:, :, None, None])
                       +       shift[:, :, None, None])
        h           = self.act(self.conv2(self.act(h)))
        return h + self.skip(x)


class SpatialAttention(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.norm  = _gn(ch)
        n_heads    = max(1, ch // 32)
        self.attn  = nn.MultiheadAttention(ch, n_heads, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H_, W_ = x.shape
        h = self.norm(x).reshape(B, C, H_ * W_).transpose(1, 2)
        h, _ = self.attn(h, h, h)
        return x + h.transpose(1, 2).reshape(B, C, H_, W_)


class ScoreUNet(nn.Module):
    """
    Geometry-conditioned UNet for diffusion noise prediction.

    Input:   [B, 5, H, W]  = 1 noisy intensity + 4 geometry channels
    Output:  [B, 1, H, W]  = predicted noise ε̂

    3-level encoder/decoder (spatial: 20 → 10 → 5 → 10 → 20).
    Skip connections:
      bottleneck (5x5) → up → 10x10  cat  enc1 (10x10)
      dec_a      (10x10) → up → 20x20  cat  enc0 (20x20)
    """
    def __init__(self, in_ch: int = 5, base_ch: int = 32,
                 time_dim: int = 128):
        super().__init__()
        ch = base_ch
        td = time_dim * 4

        self.time_embed = TimestepEmbedding(time_dim)

        # Encoder
        self.enc0  = ResBlock(in_ch, ch,    td)   # [B, ch,   20, 20]
        self.enc1  = ResBlock(ch,    ch*2,  td)   # [B, ch*2, 10, 10]
        self.enc2  = ResBlock(ch*2,  ch*4,  td)   # [B, ch*4,  5,  5]
        self.down1 = nn.AvgPool2d(2)
        self.down2 = nn.AvgPool2d(2)

        # Bottleneck
        self.mid1  = ResBlock(ch*4, ch*4, td)
        self.mid_a = SpatialAttention(ch*4)
        self.mid2  = ResBlock(ch*4, ch*4, td)

        # Decoder
        # dec_a:  up(bottleneck, 5→10) + enc1(10)  →  ch*2
        self.dec_a = ResBlock(ch*4 + ch*2, ch*2, td)
        # dec_b:  up(dec_a, 10→20)    + enc0(20)  →  ch
        self.dec_b = ResBlock(ch*2 + ch,   ch,   td)

        self.up2   = nn.Upsample(scale_factor=2, mode='bilinear',
                                 align_corners=False)
        self.up1   = nn.Upsample(scale_factor=2, mode='bilinear',
                                 align_corners=False)
        self.out   = nn.Sequential(_gn(ch), nn.SiLU(),
                                   nn.Conv2d(ch, 1, 1))

    def forward(self, xt: torch.Tensor, t: torch.Tensor,
                geometry: torch.Tensor) -> torch.Tensor:
        x   = torch.cat([xt, geometry], dim=1)      # [B, 5, 20, 20]
        te  = self.time_embed(t)                     # [B, td]

        e0  = self.enc0(x,             te)           # [B, ch,   20, 20]
        e1  = self.enc1(self.down1(e0), te)          # [B, ch*2, 10, 10]
        e2  = self.enc2(self.down2(e1), te)          # [B, ch*4,  5,  5]

        h   = self.mid1(e2, te)
        h   = self.mid_a(h)
        h   = self.mid2(h,  te)                      # [B, ch*4,  5,  5]

        h   = self.dec_a(
                  torch.cat([self.up2(h), e1], dim=1), te)  # [B,ch*2,10,10]
        h   = self.dec_b(
                  torch.cat([self.up1(h), e0], dim=1), te)  # [B,ch,  20,20]

        return self.out(h)                           # [B, 1, 20, 20]


# Instantiate and verify
model = ScoreUNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ScoreUNet parameters: {n_params:,}')

with torch.no_grad():
    _test = model(
        torch.randn(2, 1, H, W).to(DEVICE),
        torch.randint(0, T_DIFF, (2,)).to(DEVICE),
        GEOMETRY.unsqueeze(0).expand(2, -1, -1, -1).to(DEVICE)
    )
assert _test.shape == (2, 1, H, W), f'Unexpected shape {_test.shape}'
print(f'Forward-pass shape: {_test.shape}  ✓')

## Section 5 — Diffusion Forward Process + Training

In [ ]:
def forward_diffuse(x0: torch.Tensor,
                    t:  torch.Tensor
                   ) -> Tuple[torch.Tensor, torch.Tensor]:
    """Add noise to x0 at timestep t.  Returns (xt, noise)."""
    noise = torch.randn_like(x0)
    ab    = ALPHA_BAR[t].to(x0.device).view(-1, 1, 1, 1)
    xt    = torch.sqrt(ab) * x0 + torch.sqrt(1.0 - ab) * noise
    return xt, noise


optimizer  = torch.optim.AdamW(model.parameters(),
                                lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_EPOCHS)
geo_dev    = GEOMETRY.unsqueeze(0).to(DEVICE)
fields_dev = FIELDS_TENSOR.to(DEVICE)

train_losses = []
print(f'\nTraining for {N_EPOCHS} epochs on {N_WINDOWS} windows...')
for epoch in range(N_EPOCHS):
    idx = torch.randperm(N_WINDOWS)
    ep_loss = []
    for start in range(0, N_WINDOWS, BATCH_SIZE):
        bi  = idx[start : start + BATCH_SIZE]
        x0  = fields_dev[bi]
        geo = geo_dev.expand(x0.shape[0], -1, -1, -1)
        t   = torch.randint(0, T_DIFF, (x0.shape[0],), device=DEVICE)
        xt, noise = forward_diffuse(x0, t)
        pred      = model(xt, t, geo)
        loss      = F.mse_loss(pred, noise)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss.append(loss.item())
    scheduler.step()
    train_losses.append(np.mean(ep_loss))
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1:3d}/{N_EPOCHS}  '
              f'loss={train_losses[-1]:.5f}  '
              f'lr={scheduler.get_last_lr()[0]:.1e}')

model.eval()
print('Training complete.')

## Section 6 — Inference: Tweedie Mean + DDIM Sampling

In [ ]:
def estimate_mean_intensity(geometry: torch.Tensor,
                             n_samples: int = 64,
                             t_eval:    int = 20) -> torch.Tensor:
    """
    Posterior mean E_λ[λ(x)] via Tweedie formula.
      x̂₀ = (xₜ − √(1−ᾱₜ) · ε̂) / √ᾱₜ
    Average over n_samples noisy realisations, clamp ≥ 0, normalise.
    Returns [1, H, W].
    """
    geo = geometry.unsqueeze(0).expand(n_samples, -1, -1, -1).to(DEVICE)
    ab  = ALPHA_BAR[t_eval].to(DEVICE)
    tv  = torch.full((n_samples,), t_eval, device=DEVICE)
    x0p = torch.randn(n_samples, 1, H, W, device=DEVICE)
    xt  = (torch.sqrt(ab) * x0p +
           torch.sqrt(1.0 - ab) * torch.randn_like(x0p))
    with torch.no_grad():
        eps  = model(xt, tv, geo)
        x0_e = (xt - torch.sqrt(1.0 - ab) * eps) / torch.sqrt(ab)
    lm = x0_e.mean(dim=0, keepdim=True).clamp(min=0)   # [1,1,H,W]
    s  = lm.sum()
    if s > 0: lm = lm / s
    return lm.squeeze(0).cpu()   # [1, H, W]


def ddim_sample(geometry: torch.Tensor,
                K:     int = 30,
                steps: int = 20) -> torch.Tensor:
    """
    Draw K intensity field samples via DDIM reverse diffusion.
    Returns [K, 1, H, W] — each sample L1-normalised and non-negative.
    """
    geo = geometry.unsqueeze(0).expand(K, -1, -1, -1).to(DEVICE)
    ts  = torch.linspace(T_DIFF - 1, 0, steps).long()
    x   = torch.randn(K, 1, H, W, device=DEVICE)

    for i, tv in enumerate(ts):
        tb  = torch.full((K,), tv.item(), device=DEVICE)
        abt = ALPHA_BAR[tv].to(DEVICE)
        abp = (ALPHA_BAR[ts[i + 1]].to(DEVICE)
               if i < len(ts) - 1 else torch.tensor(1.0, device=DEVICE))
        with torch.no_grad():
            eps = model(x, tb, geo)
        x0p = ((x - torch.sqrt(1.0 - abt) * eps) /
               torch.sqrt(abt)).clamp(-3, 3)
        x   = torch.sqrt(abp) * x0p + torch.sqrt(1.0 - abp) * eps

    s    = x.clamp(min=0)
    sums = s.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1e-8)
    return (s / sums).cpu()   # [K, 1, H, W]


print('\nEstimating mean intensity...')
LAMBDA_MEAN       = estimate_mean_intensity(GEOMETRY)       # [1, H, W]
LAMBDA_MEAN_FLAT  = LAMBDA_MEAN[0].view(-1)                 # [H*W]

print('Drawing DDIM posterior samples...')
LAMBDA_SAMPLES     = ddim_sample(GEOMETRY, K=30, steps=20)  # [30, 1, H, W]
LAMBDA_SAMPLES_FLAT = LAMBDA_SAMPLES[:, 0, :, :].view(30, -1)  # [30, H*W]

LAMBDA_HIST       = FIELDS_TENSOR.mean(dim=0).cpu()         # [1, H, W]
LAMBDA_HIST_FLAT  = LAMBDA_HIST[0].view(-1)                 # [H*W]

print(f'  Mean intensity range: '
      f'[{LAMBDA_MEAN_FLAT.min():.4f}, {LAMBDA_MEAN_FLAT.max():.4f}]')

## Section 7 — Camera Model

In [ ]:
def check_line_of_sight(sensor_loc: torch.Tensor,
                         x_grid:    torch.Tensor,
                         n_steps:   int = 10) -> torch.Tensor:
    """
    Ray-march from sensor to each grid cell.
    Returns [H*W] float tensor: 1 = clear, 0 = blocked by wall.
    """
    los = torch.ones(x_grid.shape[0])
    for step in range(1, n_steps):
        tf   = step / n_steps
        ip   = sensor_loc[None] + tf * (x_grid - sensor_loc[None])
        jj   = ((ip[:, 0] + HALF) / RESOLUTION).long().clamp(0, W - 1)
        ii   = ((ip[:, 1] + HALF) / RESOLUTION).long().clamp(0, H - 1)
        los  = los * (SDF_MAP[ii, jj] > 0).float()
    return los


def detection_probability(x_grid:    torch.Tensor,
                           sensor_loc: torch.Tensor,
                           angle_deg:  torch.Tensor) -> torch.Tensor:
    """
    γ(l, aᵢ) at each grid cell for camera at sensor_loc facing angle_deg.
    Returns [H*W].
    """
    delta      = x_grid - sensor_loc[None]
    dist       = torch.norm(delta, dim=-1)
    cell_angle = torch.atan2(delta[:, 1], delta[:, 0]) * 180.0 / pi
    angle_diff = (cell_angle - angle_deg).abs() % 360.0
    angle_diff = torch.minimum(angle_diff, 360.0 - angle_diff)

    range_factor = torch.exp(-dist ** 2 / (2 * (MAX_RANGE / 2) ** 2))
    fov_factor   = (angle_diff < FOV_DEG / 2).float()
    los          = check_line_of_sight(sensor_loc, x_grid)

    return RHO * range_factor * fov_factor * los   # [H*W]


def miss_probability_network(x_grid:  torch.Tensor,
                              sensors: torch.Tensor,
                              angles:  torch.Tensor) -> torch.Tensor:
    """
    π_C(l, a) = ∏ᵢ (1 − γ(l, aᵢ)) — joint miss probability.
    Returns [H*W].
    """
    pi = torch.ones(H * W)
    for i in range(sensors.shape[0]):
        pi = pi * (1.0 - detection_probability(x_grid, sensors[i], angles[i]))
    return pi


def void_probability_mc(sensors: torch.Tensor,
                         angles:  torch.Tensor,
                         lambda_samples_flat: torch.Tensor) -> torch.Tensor:
    """
    Full void probability (Eq. 2) via Monte Carlo over λ samples.
    P(N̄=0) ≈ (1/K) Σ_k exp(−∫ λ_k(l) π(l,a) dl)
    """
    pi      = miss_probability_network(GRID_CENTRES, sensors, angles)
    thinned = (lambda_samples_flat * pi[None, :] * DX2).sum(dim=-1)
    return torch.exp(-thinned).mean()


def void_probability_jensen(sensors: torch.Tensor,
                              angles:  torch.Tensor,
                              lambda_mean_flat: torch.Tensor) -> torch.Tensor:
    """
    Jensen lower bound (Eq. 3) — uses mean intensity only.
    ν(a) = exp(−∫ E[λ(l)] π(l,a) dl)
    """
    pi      = miss_probability_network(GRID_CENTRES, sensors, angles)
    thinned = (lambda_mean_flat * pi * DX2).sum()
    return torch.exp(-thinned)

## Section 8 — Greedy Camera Initialisation

In [ ]:
def greedy_initialise(lambda_mean_flat: torch.Tensor,
                       M:           int   = N_CAMERAS,
                       angle_step:  float = 45.0
                      ) -> Tuple[torch.Tensor, torch.Tensor, List]:
    """
    Place M cameras one at a time, each maximising the Jensen lower
    bound void probability over all candidate (location, angle) pairs.
    Returns sensors [M,2], angles [M], void-probability history.
    """
    valid_mask  = SDF_MAP.view(-1) > 0.08
    valid_locs  = GRID_CENTRES[valid_mask]
    cand_angles = torch.arange(0, 360, angle_step)

    placed_sensors = []
    placed_angles  = []
    history        = []
    current_pi     = torch.ones(H * W)

    print(f'  Candidate locations: {valid_locs.shape[0]}  '
          f'Angles: {len(cand_angles)}')

    for m in range(M):
        best_vp    = -inf
        best_loc   = None
        best_angle = None

        for loc in valid_locs:
            for ang in cand_angles:
                gamma  = detection_probability(GRID_CENTRES, loc, ang)
                pi_new = current_pi * (1.0 - gamma)
                vp     = torch.exp(
                    -(lambda_mean_flat * pi_new * DX2).sum()
                ).item()
                if vp > best_vp:
                    best_vp    = vp
                    best_loc   = loc.clone()
                    best_angle = ang.clone()

        placed_sensors.append(best_loc)
        placed_angles.append(best_angle)
        history.append(best_vp)
        current_pi = current_pi * (
            1.0 - detection_probability(GRID_CENTRES, best_loc, best_angle)
        )
        print(f'    Camera {m+1}: '
              f'({best_loc[0].item():.1f}, {best_loc[1].item():.1f}) m  '
              f'{best_angle.item():.0f}°   VP={best_vp:.4f}')

    return torch.stack(placed_sensors), torch.stack(placed_angles), history


print('\n=== Greedy — Score Model mean ===')
SG, AG, GH = greedy_initialise(LAMBDA_MEAN_FLAT)

print('\n=== Greedy — Histogram mean ===')
SH, AH, HH = greedy_initialise(LAMBDA_HIST_FLAT)

## Section 9 — Gradient-Based Refinement

In [ ]:
def optimise_cameras(sensors_init: torch.Tensor,
                      angles_init:  torch.Tensor,
                      lambda_samples_flat: torch.Tensor,
                      n_iter: int   = 120,
                      lr:     float = 0.25,
                      label:  str   = ''
                     ) -> Tuple[torch.Tensor, torch.Tensor, List]:
    """
    Gradient ascent on the full Monte Carlo void probability.
    Jointly optimises sensor positions (x,y) and orientations (degrees).
    SDF constraint prevents sensors moving into walls.
    """
    sensors = sensors_init.clone().detach().requires_grad_(True)
    angles  = angles_init.clone().detach().float().requires_grad_(True)

    opt   = torch.optim.Adam([sensors, angles], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_iter)
    hist  = []

    for it in range(n_iter):
        opt.zero_grad()
        vp   = void_probability_mc(sensors, angles, lambda_samples_flat)
        loss = -vp
        loss.backward()

        with torch.no_grad():
            # Freeze gradient for sensors that would move into a wall
            for mi in range(sensors.shape[0]):
                ri, rj = world_to_idx(sensors[mi, 0].item(),
                                       sensors[mi, 1].item())
                if SDF_MAP[ri, rj].item() < 0.05:
                    if sensors.grad is not None:
                        sensors.grad[mi] *= 0.0
            angles.data = angles.data % 360.0

        opt.step()
        sched.step()

        with torch.no_grad():
            sensors.data = sensors.data.clamp(-HALF + 1.0, HALF - 1.0)

        hist.append(vp.item())
        if (it + 1) % 60 == 0:
            print(f'  [{label}] iter {it+1:3d}   VP={vp.item():.4f}')

    return sensors.detach(), angles.detach(), hist


print('\n=== Gradient Refinement — Score Model ===')
SO, AO, OH  = optimise_cameras(SG, AG, LAMBDA_SAMPLES_FLAT, label='Score')

print('\n=== Gradient Refinement — Histogram ===')
SHO, AHO, HHO = optimise_cameras(SH, AH, LAMBDA_SAMPLES_FLAT, label='Hist')

# Compute final void probabilities
vp_sg = void_probability_mc(SG,  AG,  LAMBDA_SAMPLES_FLAT).item()
vp_so = void_probability_mc(SO,  AO,  LAMBDA_SAMPLES_FLAT).item()
vp_hg = void_probability_mc(SH,  AH,  LAMBDA_SAMPLES_FLAT).item()
vp_ho = void_probability_mc(SHO, AHO, LAMBDA_SAMPLES_FLAT).item()

print(f'\n{"="*52}')
print(f'{"Method":<24}  {"Void Prob (MC)":>14}')
print(f'{"="*52}')
for name, vp in [('Score + Greedy',  vp_sg), ('Score + Refined', vp_so),
                  ('Hist  + Greedy',  vp_hg), ('Hist  + Refined', vp_ho)]:
    print(f'{name:<24}  {vp:>14.4f}')
print(f'{"="*52}')

## Section 10 — Simulation Study

In [ ]:
def simulation_study(placements: dict, n_trials: int = 500) -> dict:
    """
    Ground-truth evaluation.
    For each trial:
      - Sample a held-out intensity field
      - Draw Poisson number of whole-trajectory events (~5–8 per trial)
      - For each trajectory, compute joint miss probability across all cameras
      - Record 1 if all trajectories detected, 0 otherwise
    Returns detection rate per method.
    """
    test_windows = [generate_window(30) for _ in range(20)]
    test_fields  = [build_intensity_field(w)[0].numpy()
                    for w in test_windows]

    results = {name: [] for name in placements}

    for _ in range(n_trials):
        lam_true = random.choice(test_fields)            # [H, W]

        # ~6 whole-trajectory events per trial
        expected  = lam_true * DX2 * 6
        n_events  = np.random.poisson(expected)          # [H, W]

        event_locs = []
        for (i, j) in zip(*np.where(n_events > 0)):
            for _ in range(n_events[i, j]):
                x = XS[j] + np.random.uniform(-RESOLUTION/2, RESOLUTION/2)
                y = YS[i] + np.random.uniform(-RESOLUTION/2, RESOLUTION/2)
                event_locs.append((x, y))

        if not event_locs:
            continue

        for name, (sens, angs) in placements.items():
            sn = sens.numpy()
            an = angs.numpy()
            all_detected = True

            for (x, y) in event_locs:
                # Joint miss probability across all cameras
                miss = 1.0
                for i in range(sn.shape[0]):
                    dx    = x - sn[i, 0]
                    dy    = y - sn[i, 1]
                    dist  = math.sqrt(dx ** 2 + dy ** 2)
                    a_deg = math.degrees(math.atan2(dy, dx))
                    adiff = abs(a_deg - an[i]) % 360
                    adiff = min(adiff, 360 - adiff)
                    g     = (RHO
                             * math.exp(-dist**2 / (2*(MAX_RANGE/2)**2))
                             * float(adiff < FOV_DEG / 2))
                    miss *= (1.0 - g)

                # Bernoulli: not detected with probability = miss
                if random.random() < miss:
                    all_detected = False
                    break

            results[name].append(1 if all_detected else 0)

    return {k: np.mean(v) for k, v in results.items()}


print('\nRunning simulation study (500 trials)...')
placements = {
    'Score+Greedy':  (SG,  AG),
    'Score+Refined': (SO,  AO),
    'Hist+Greedy':   (SH,  AH),
    'Hist+Refined':  (SHO, AHO),
}
sim_results = simulation_study(placements, n_trials=500)

vp_map = {'Score+Greedy': vp_sg, 'Score+Refined': vp_so,
           'Hist+Greedy':  vp_hg, 'Hist+Refined':  vp_ho}

print(f'\n{"="*58}')
print(f'{"Method":<22}  {"VP (MC)":>10}  {"Sim Det Rate":>12}')
print(f'{"="*58}')
for name in placements:
    print(f'{name:<22}  {vp_map[name]:>10.4f}  {sim_results[name]:>12.4f}')
print(f'{"="*58}')

## Section 11 — Figures

In [ ]:
# ── Fig 1: Training loss ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(train_losses, lw=2, color='#2196F3', marker='o', ms=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Diffusion Model Training Loss'); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/01_training_loss.png', dpi=120, bbox_inches='tight')
plt.close(); print('Saved 01_training_loss.png')

# ── Fig 2: Sample trajectories ────────────────────────────────
TC = {'NS':'#2196F3','EW':'#F44336','diagonal':'#4CAF50',
      'central':'#FF9800','corner':'#9C27B0'}
fig, ax = plt.subplots(figsize=(7, 7))
ax.add_patch(patches.Rectangle((-HALF,-HALF), MALL_SIZE, MALL_SIZE,
                                lw=2, ec='black', fc='#f5f5f5'))
for t in ALL_WINDOWS[0]:
    ax.plot(t.points[:,0], t.points[:,1],
            color=TC[t.pattern], alpha=.4, lw=1.0)
for nm, pos in DOORS.items():
    ax.plot(*pos, 'ks', ms=9, zorder=5)
    ax.annotate(f'Door {nm}', pos, textcoords='offset points',
                xytext=(5,5), fontsize=9)
ax.legend(handles=[Line2D([0],[0],color=c,label=p,lw=2)
                   for p,c in TC.items()], loc='lower left', fontsize=8)
ax.set_xlim(-HALF-1, HALF+1); ax.set_ylim(-HALF-1, HALF+1)
ax.set_aspect('equal')
ax.set_title(f'Sample Window — {len(ALL_WINDOWS[0])} trajectories',
             fontsize=12)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/02_trajectories.png', dpi=120, bbox_inches='tight')
plt.close(); print('Saved 02_trajectories.png')

# ── Fig 3: Intensity comparison ───────────────────────────────
fig, axs = plt.subplots(1, 3, figsize=(14, 5))
true_m = FIELDS_TENSOR.mean(dim=0)[0].numpy()
for ax_, arr, ttl in zip(axs,
    [true_m, LAMBDA_MEAN[0].numpy(), LAMBDA_HIST[0].numpy()],
    ['True Mean\n(all data)',
     'Score Model\n(Tweedie mean)',
     'Histogram\nBaseline']):
    im = ax_.imshow(arr, origin='lower',
                    extent=[-HALF, HALF, -HALF, HALF], cmap='hot')
    ax_.set_title(ttl, fontsize=11)
    plt.colorbar(im, ax=ax_, fraction=.046)
    ax_.set_xlabel('x (m)'); ax_.set_ylabel('y (m)')
plt.suptitle('Intensity Field Estimates', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_intensity_comparison.png',
            dpi=120, bbox_inches='tight')
plt.close(); print('Saved 03_intensity_comparison.png')

# ── Fig 4: Detection map (single test camera) ─────────────────
test_d = detection_probability(GRID_CENTRES,
                                torch.tensor([0., 0.]),
                                torch.tensor(45.)).view(H, W).numpy()
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(test_d, origin='lower',
               extent=[-HALF, HALF, -HALF, HALF],
               cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, label='Detection probability')
ax.plot(0, 0, '^', color='blue', ms=12, mec='black', label='Camera')
ax.set_title('Detection Map — Centre Camera, 45°')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/04_detection_map.png', dpi=120, bbox_inches='tight')
plt.close(); print('Saved 04_detection_map.png')

# ── Fig 5: 2×2 camera placement grid ─────────────────────────
def draw_camera(ax_, loc, ang_deg, col):
    x, y = loc
    ar   = math.radians(ang_deg)
    hf   = math.radians(FOV_DEG / 2)
    r    = MAX_RANGE
    xs   = [x, x+r*math.cos(ar+hf), x+r*math.cos(ar-hf), x]
    ys   = [y, y+r*math.sin(ar+hf), y+r*math.sin(ar-hf), y]
    ax_.fill(xs, ys, color=col, alpha=.15)
    ax_.plot(xs, ys, color=col, lw=.8, alpha=.6)
    ax_.plot(x, y, '^', color=col, ms=11, mec='black', mew=.5, zorder=6)

fig, axes = plt.subplots(2, 2, figsize=(14, 12), facecolor='#0d0d1a')
cfgs = [
    ('Score + Greedy',  SG,  AG,  '#FF5722', vp_sg, LAMBDA_MEAN[0].numpy()),
    ('Score + Refined', SO,  AO,  '#FF5722', vp_so, LAMBDA_MEAN[0].numpy()),
    ('Hist  + Greedy',  SH,  AH,  '#29B6F6', vp_hg, LAMBDA_HIST[0].numpy()),
    ('Hist  + Refined', SHO, AHO, '#29B6F6', vp_ho, LAMBDA_HIST[0].numpy()),
]
for ax_, (ttl, sen, ang, col, vp, lf) in zip(axes.ravel(), cfgs):
    ax_.imshow(lf, origin='lower',
               extent=[-HALF, HALF, -HALF, HALF],
               cmap='hot', alpha=.7)
    ax_.add_patch(patches.Rectangle(
        (-HALF, -HALF), MALL_SIZE, MALL_SIZE,
        lw=2, ec='white', fc='none'))
    for t in ALL_WINDOWS[0][:25]:
        ax_.plot(t.points[:,0], t.points[:,1],
                 color='cyan', alpha=.18, lw=.8)
    for pos in DOORS.values():
        ax_.plot(*pos, 'ws', ms=7, zorder=5, mec='black', mew=.8)
    for i, (sl, sa) in enumerate(zip(sen.numpy(), ang.numpy())):
        draw_camera(ax_, sl, sa, col)
        ax_.annotate(f'C{i+1}', sl, fontsize=9,
                     color='white', fontweight='bold',
                     xytext=(3, 3), textcoords='offset points')
    ax_.set_xlim(-HALF-.5, HALF+.5)
    ax_.set_ylim(-HALF-.5, HALF+.5)
    ax_.set_title(f'{ttl}   VP={vp:.4f}',
                  fontsize=10, color='white', pad=5)
    ax_.set_facecolor('#1a1a2e')
    ax_.tick_params(colors='white')
    for sp in ax_.spines.values(): sp.set_edgecolor('white')
    ax_.set_xlabel('x (m)', color='white', fontsize=8)
    ax_.set_ylabel('y (m)', color='white', fontsize=8)
fig.suptitle(
    'Mall Camera Placement — Score Model vs Histogram Baseline',
    fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/05_camera_placements.png',
            dpi=140, bbox_inches='tight', facecolor='#0d0d1a')
plt.close(); print('Saved 05_camera_placements.png')

# ── Fig 6: Convergence plot ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(OH,  color='#FF5722', lw=2, label='Score refined')
ax.plot(HHO, color='#29B6F6', lw=2, label='Hist refined')
ax.axhline(vp_sg, color='#FF5722', ls='--', alpha=.5,
           label=f'Score greedy  {vp_sg:.4f}')
ax.axhline(vp_hg, color='#29B6F6', ls='--', alpha=.5,
           label=f'Hist greedy   {vp_hg:.4f}')
ax.set_xlabel('Optimisation Iteration')
ax.set_ylabel('Void Probability (MC)')
ax.set_title('Gradient Refinement Convergence')
ax.legend(fontsize=9); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/06_convergence.png', dpi=120, bbox_inches='tight')
plt.close(); print('Saved 06_convergence.png')

# ── Fig 7: Evaluation bar chart ───────────────────────────────
methods = list(placements.keys())
cols_b  = ['#FF5722','#FF5722','#29B6F6','#29B6F6']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax_, vals, ttl, yl in [
    (axes[0], [vp_map[m] for m in methods],
     'Monte Carlo Void Probability', 'Void Probability'),
    (axes[1], [sim_results[m] for m in methods],
     'Simulation Detection Rate',
     'Fraction of Trials — All Detected')
]:
    mx = max(vals) if max(vals) > 0 else 1.0
    bars = ax_.bar(methods, vals, color=cols_b, alpha=.85, edgecolor='black')
    ax_.set_title(ttl, fontsize=12); ax_.set_ylabel(yl)
    ax_.set_ylim(0, mx * 1.3)
    ax_.tick_params(axis='x', rotation=12)
    for bar, v in zip(bars, vals):
        ax_.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + mx * 0.02,
                 f'{v:.4f}', ha='center', va='bottom', fontsize=9)
axes[0].legend(
    handles=[Patch(color='#FF5722', label='Score model'),
             Patch(color='#29B6F6', label='Histogram')],
    fontsize=9)
plt.suptitle('Evaluation Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/07_evaluation.png', dpi=120, bbox_inches='tight')
plt.close(); print('Saved 07_evaluation.png')

print(f'\nAll outputs written to {OUT_DIR}/')
print('PIPELINE COMPLETE.')